[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/064.%20The%20Cross-Docking%20Door%20Assignment%20Problem/P64-Tier-3_executed.ipynb)

---


# 64. The Cross-Docking Door Assignment Problem
## Tier 3: Cuckoo Search with Lévy Flights

### Purpose and scope

This notebook uses Cuckoo Search, a population-based metaheuristic, to seek a low-cost assignment of origins to inbound doors and destinations to outbound doors. It reports the best solution found by a finite stochastic search; it does not prove that this solution is globally optimal. The notebook also examines convergence, Lévy-flight step sizes, descriptive comparisons with random assignments and Hill Climbing, and sensitivity to selected algorithm parameters.

The instance contains a flow volume from each origin to each destination, a distance between each inbound and outbound door, and capacity limits for every door. Each origin is assigned to exactly one inbound door, and each destination to exactly one outbound door. Assignments are one-to-one: no door serves more than one origin or destination. Door capacities are therefore checked against the volume or demand of the assigned entity. The example's capacities are large enough for every individual origin and destination, but the capacity checks remain part of the model.

The objective is total volume-weighted door-to-door distance. It is a transportation-work measure, not a monetary cost: no monetary rate per unit of distance is supplied. The model treats the supplied door-to-door distances as fixed and does not model handling time, congestion, travel routes within the facility, multiple time periods, or uncertainty in volumes.

### Model

Let \(O\) be the set of origins, \(D\) the set of destinations, \(I\) the set of inbound doors, and \(J\) the set of outbound doors. Let \(v_{od}\) be the flow volume from origin \(o\) to destination \(d\), and let \(c_{ij}\) be the distance from inbound door \(i\) to outbound door \(j\). An assignment consists of mappings \(a: O \to I\) and \(b: D \to J\). The objective evaluated by `calculate_nest_fitness` is

$$
F(a,b) = \sum_{o \in O}\sum_{d \in D} v_{od}\,c_{a(o),b(d)}.
$$

Every origin and destination is assigned once. One-to-one assignments require \(a(o_1) \ne a(o_2)\) for distinct origins and \(b(d_1) \ne b(d_2)\) for distinct destinations. The capacity conditions are \(V_o \leq C_i^{\mathrm{in}}\) whenever \(a(o)=i\), and \(Q_d \leq C_j^{\mathrm{out}}\) whenever \(b(d)=j\), where \(V_o=\sum_{d\in D}v_{od}\), \(Q_d=\sum_{o\in O}v_{od}\), and the \(C\)'s are door capacities. `generate_random_nest` constructs one-to-one assignments; `calculate_nest_fitness` validates assignments and capacities before evaluating the objective.

The fitness is minimized. Since this is a discrete assignment problem, a Lévy-flight step cannot be applied as a continuous coordinate change. Here its absolute magnitude, multiplied by `step_scale` and capped at one, sets the probability of making a discrete reassignment move. A move changes an assignment while preserving its one-to-one structure; the unused door, when one exists, can be used for a reassignment.

### Cuckoo Search in this implementation

A nest is one feasible pair of door assignments. The population is a collection of nests. At each iteration, the algorithm selects a population nest, generates a candidate through Lévy-flight-guided discrete moves, and compares that candidate with a randomly selected nest. The candidate replaces the selected nest only when its fitness is lower. After candidate generation, a fraction of the worst nests is abandoned and replaced with random feasible nests. The best solution seen at any point is retained separately, so abandoning a population member cannot erase the recorded best-so-far solution.

For a nest \(x\), let \(F(x)\) denote its fitness, and let \(x^*\) be the best nest seen so far. The update is

$$
x^*_{t+1} = \begin{cases}
x_{t+1}, & F(x_{t+1}) < F(x^*_t), \\
x^*_t, & \text{otherwise}.
\end{cases}
$$

The population update, abandonment, and best-so-far tracking are implemented in `cuckoo_search`. `levy_flight` uses Mantegna's construction for \(0<\beta<2\); at \(\beta=2\), it uses the corresponding Gaussian limit. The step-size distribution has a heavy tail for the selected \(\beta=1.5\), so large moves are possible alongside small moves. In this discrete implementation, that distribution influences move probability; it does not directly move a solution by a continuous distance.

The search has no optimality guarantee. With \(|O|\) origins, \(|D|\) destinations, \(|I|\) inbound doors, and \(|J|\) outbound doors, the number of one-to-one assignments is

$$
\frac{|I|!}{(|I|-|O|)!}\,\frac{|J|!}{(|J|-|D|)!},
$$

when there are at least as many doors as assigned entities. This combinatorial growth motivates a population-based search for larger instances; it does not establish that this metaheuristic will outperform another method on every instance.

### Reading the experiments

The main run uses 25 nests, 500 iterations, abandonment probability 0.25, Lévy parameter \(\beta=1.5\), and step scale 0.1. These are algorithm settings, not guarantees of a particular cost or convergence iteration. Results depend on the random sequence. A single run is illustrative; the random and Hill Climbing comparisons are descriptive, and the sensitivity experiments show outcomes for the specified settings rather than establishing universal parameter recommendations.

The plots show best-so-far and population-average fitness, a fitness-based population-diversity measure, sampled Lévy step magnitudes, and the final population's fitness distribution. Fitness diversity is the coefficient of variation of population fitnesses; it measures spread in objective values, not the number of distinct assignment patterns. The notebook does not report an exact optimum or a statistical significance test based on a single Cuckoo Search run.

In [ ]:
from dataclasses import dataclass
import math
import random
import time
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Keeping both random generators seeded makes the stochastic experiments reproducible.
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class CrossDockInstance:
    """Input data and capacity limits for a cross-docking instance."""

    origins: List[str]
    destinations: List[str]
    inbound_doors: List[str]
    outbound_doors: List[str]
    distance_matrix: np.ndarray
    volume_matrix: np.ndarray
    origin_volumes: np.ndarray
    destination_demands: np.ndarray
    inbound_capacities: np.ndarray
    outbound_capacities: np.ndarray


@dataclass
class Nest:
    """One feasible pair of origin and destination door assignments."""

    origin_assignments: Dict[str, str]
    destination_assignments: Dict[str, str]
    fitness: float
    age: int = 0


def create_tier3_example() -> CrossDockInstance:
    """Create the specified five-origin, four-destination example."""
    origins = ['O1', 'O2', 'O3', 'O4', 'O5']
    destinations = ['D1', 'D2', 'D3', 'D4']
    inbound_doors = ['I1', 'I2', 'I3', 'I4', 'I5', 'I6']
    outbound_doors = ['O1', 'O2', 'O3', 'O4', 'O5']

    distance_matrix = np.array([
        [10, 15, 22, 28, 35],
        [12, 8, 18, 25, 32],
        [18, 14, 10, 20, 28],
        [25, 20, 15, 12, 22],
        [32, 28, 22, 18, 10],
        [40, 35, 30, 25, 15],
    ])

    volume_matrix = np.array([
        [35, 25, 30, 20],
        [28, 40, 22, 25],
        [30, 20, 45, 25],
        [25, 30, 25, 40],
        [20, 25, 30, 35],
    ])

    origin_volumes = np.sum(volume_matrix, axis=1)
    destination_demands = np.sum(volume_matrix, axis=0)
    inbound_capacities = np.array([150, 150, 150, 150, 150, 150])
    outbound_capacities = np.array([180, 180, 180, 180, 180])

    return CrossDockInstance(
        origins=origins,
        destinations=destinations,
        inbound_doors=inbound_doors,
        outbound_doors=outbound_doors,
        distance_matrix=distance_matrix,
        volume_matrix=volume_matrix,
        origin_volumes=origin_volumes,
        destination_demands=destination_demands,
        inbound_capacities=inbound_capacities,
        outbound_capacities=outbound_capacities,
    )


instance = create_tier3_example()

assert instance.distance_matrix.shape == (
    len(instance.inbound_doors), len(instance.outbound_doors)
)
assert instance.volume_matrix.shape == (
    len(instance.origins), len(instance.destinations)
)
assert np.all(instance.origin_volumes <= np.max(instance.inbound_capacities))
assert np.all(instance.destination_demands <= np.max(instance.outbound_capacities))

print('Cross-docking instance created.')
print(f'Origins: {instance.origins}')
print(f'Destinations: {instance.destinations}')
print(f'Inbound doors: {instance.inbound_doors}')
print(f'Outbound doors: {instance.outbound_doors}')
print('\nDistance matrix (inbound doors by outbound doors):')
print(pd.DataFrame(
    instance.distance_matrix,
    index=instance.inbound_doors,
    columns=instance.outbound_doors,
))
print('\nVolume matrix (origins by destinations):')
print(pd.DataFrame(
    instance.volume_matrix,
    index=instance.origins,
    columns=instance.destinations,
))

In [ ]:
def calculate_nest_fitness(nest: Nest, instance: CrossDockInstance) -> float:
    """Validate a nest and return its total volume-weighted distance."""
    if set(nest.origin_assignments) != set(instance.origins):
        raise ValueError('Every origin must have exactly one door assignment.')
    if set(nest.destination_assignments) != set(instance.destinations):
        raise ValueError('Every destination must have exactly one door assignment.')

    inbound_doors = list(nest.origin_assignments.values())
    outbound_doors = list(nest.destination_assignments.values())
    if len(set(inbound_doors)) != len(inbound_doors):
        raise ValueError('Each inbound door can serve at most one origin.')
    if len(set(outbound_doors)) != len(outbound_doors):
        raise ValueError('Each outbound door can serve at most one destination.')
    if not set(inbound_doors).issubset(instance.inbound_doors):
        raise ValueError('An origin is assigned to an unknown inbound door.')
    if not set(outbound_doors).issubset(instance.outbound_doors):
        raise ValueError('A destination is assigned to an unknown outbound door.')

    inbound_indices = {
        door: index for index, door in enumerate(instance.inbound_doors)
    }
    outbound_indices = {
        door: index for index, door in enumerate(instance.outbound_doors)
    }
    origin_indices = {origin: index for index, origin in enumerate(instance.origins)}
    destination_indices = {
        destination: index
        for index, destination in enumerate(instance.destinations)
    }

    for origin, door in nest.origin_assignments.items():
        door_index = inbound_indices[door]
        origin_index = origin_indices[origin]
        if instance.origin_volumes[origin_index] > instance.inbound_capacities[door_index]:
            raise ValueError(f'Inbound capacity is exceeded at door {door}.')

    for destination, door in nest.destination_assignments.items():
        door_index = outbound_indices[door]
        destination_index = destination_indices[destination]
        if instance.destination_demands[destination_index] > instance.outbound_capacities[door_index]:
            raise ValueError(f'Outbound capacity is exceeded at door {door}.')

    total_cost = 0.0
    for origin_index, origin in enumerate(instance.origins):
        inbound_index = inbound_indices[nest.origin_assignments[origin]]
        for destination_index, destination in enumerate(instance.destinations):
            outbound_index = outbound_indices[nest.destination_assignments[destination]]
            volume = instance.volume_matrix[origin_index, destination_index]
            distance = instance.distance_matrix[inbound_index, outbound_index]
            total_cost += volume * distance

    return float(total_cost)


def generate_random_nest(instance: CrossDockInstance) -> Nest:
    """Generate a uniformly sampled one-to-one assignment from available doors."""
    if len(instance.inbound_doors) < len(instance.origins):
        raise ValueError('There must be at least one inbound door per origin.')
    if len(instance.outbound_doors) < len(instance.destinations):
        raise ValueError('There must be at least one outbound door per destination.')

    selected_inbound = random.sample(instance.inbound_doors, len(instance.origins))
    selected_outbound = random.sample(
        instance.outbound_doors, len(instance.destinations)
    )
    origin_assignments = dict(zip(instance.origins, selected_inbound))
    destination_assignments = dict(zip(instance.destinations, selected_outbound))
    nest = Nest(origin_assignments, destination_assignments, fitness=0.0)
    nest.fitness = calculate_nest_fitness(nest, instance)
    return nest


def levy_flight(beta: float = 1.5) -> float:
    """Draw a scalar Lévy-flight step using Mantegna's construction."""
    if not 0 < beta <= 2:
        raise ValueError('The Lévy parameter beta must satisfy 0 < beta <= 2.')
    if math.isclose(beta, 2.0):
        return float(np.random.normal(0.0, math.sqrt(2.0)))

    sigma_u = (
        math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
        / (
            math.gamma((1 + beta) / 2)
            * beta
            * 2 ** ((beta - 1) / 2)
        )
    ) ** (1 / beta)
    u = np.random.normal(0.0, sigma_u)
    v = np.random.normal(0.0, 1.0)
    return float(u / (abs(v) ** (1 / beta)))


def _move_assignment(
    assignments: Dict[str, str], doors: List[str], movement_intensity: float
) -> None:
    """Apply a feasible reassignment or swap when a Lévy move is selected."""
    if random.random() >= movement_intensity or len(assignments) < 2:
        return

    entities = list(assignments)
    used_doors = set(assignments.values())
    unused_doors = [door for door in doors if door not in used_doors]

    if unused_doors and random.random() < 0.5:
        entity = random.choice(entities)
        assignments[entity] = random.choice(unused_doors)
    else:
        entity_1, entity_2 = random.sample(entities, 2)
        assignments[entity_1], assignments[entity_2] = (
            assignments[entity_2], assignments[entity_1]
        )


def cuckoo_movement(
    nest: Nest,
    instance: CrossDockInstance,
    beta: float = 1.5,
    step_scale: float = 0.1,
) -> Nest:
    """Create a feasible candidate using Lévy-guided discrete assignment moves."""
    if step_scale < 0:
        raise ValueError('The step scale must be non-negative.')

    new_nest = Nest(
        origin_assignments=nest.origin_assignments.copy(),
        destination_assignments=nest.destination_assignments.copy(),
        fitness=0.0,
        age=nest.age,
    )

    origin_intensity = min(abs(levy_flight(beta)) * step_scale, 1.0)
    destination_intensity = min(abs(levy_flight(beta)) * step_scale, 1.0)
    _move_assignment(
        new_nest.origin_assignments, instance.inbound_doors, origin_intensity
    )
    _move_assignment(
        new_nest.destination_assignments,
        instance.outbound_doors,
        destination_intensity,
    )
    new_nest.fitness = calculate_nest_fitness(new_nest, instance)
    return new_nest


test_nest = generate_random_nest(instance)
new_nest = cuckoo_movement(test_nest, instance)
print('Component checks:')
print(f'Random nest fitness: {test_nest.fitness:.0f} volume-distance units')
print(f'Origin assignments: {test_nest.origin_assignments}')
print(f'Destination assignments: {test_nest.destination_assignments}')
print(f'Example Lévy-flight steps: {[round(levy_flight(), 3) for _ in range(5)]}')
print(f'Candidate fitness: {new_nest.fitness:.0f} volume-distance units')

In [ ]:
def calculate_population_diversity(population: List[Nest]) -> float:
    """Return the coefficient of variation of population fitness values."""
    if len(population) <= 1:
        return 0.0
    fitnesses = np.array([nest.fitness for nest in population], dtype=float)
    mean_fitness = float(np.mean(fitnesses))
    if math.isclose(mean_fitness, 0.0):
        return 0.0
    return float(np.std(fitnesses) / abs(mean_fitness))


def cuckoo_search(
    instance: CrossDockInstance,
    population_size: int = 25,
    max_iterations: int = 500,
    abandonment_prob: float = 0.25,
    beta: float = 1.5,
    step_scale: float = 0.1,
    verbose: bool = True,
) -> Tuple[Nest, Dict]:
    """Run Cuckoo Search and return the best feasible nest and run records."""
    if population_size < 1:
        raise ValueError('Population size must be at least one.')
    if max_iterations < 1:
        raise ValueError('Maximum iterations must be at least one.')
    if not 0 <= abandonment_prob <= 1:
        raise ValueError('Abandonment probability must lie between zero and one.')
    if step_scale < 0:
        raise ValueError('The step scale must be non-negative.')

    if verbose:
        print('=' * 60)
        print('CUCKOO SEARCH WITH LÉVY FLIGHTS')
        print('=' * 60)
        print(f'Population size: {population_size}')
        print(f'Maximum iterations: {max_iterations}')
        print(f'Abandonment probability: {abandonment_prob}')
        print(f'Lévy parameter beta: {beta}')
        print(f'Step scale: {step_scale}')
        print(
            f'Problem size: {len(instance.origins)} origins, '
            f'{len(instance.destinations)} destinations, '
            f'{len(instance.inbound_doors)} inbound doors, '
            f'{len(instance.outbound_doors)} outbound doors'
        )

    population = [generate_random_nest(instance) for _ in range(population_size)]
    population.sort(key=lambda nest: nest.fitness)
    best_nest = population[0]
    best_fitness_history = [best_nest.fitness]
    avg_fitness_history = [float(np.mean([n.fitness for n in population]))]
    diversity_history = [calculate_population_diversity(population)]
    levy_step_history = []
    improvements = 0
    abandonments = 0
    start_time = time.time()

    if verbose:
        print(f'Initial best fitness: {best_nest.fitness:.0f}')
        print(f'Initial average fitness: {avg_fitness_history[0]:.0f}')
        print(f'Initial fitness diversity: {diversity_history[0]:.3f}')

    for iteration in range(max_iterations):
        iteration_improvements = 0
        iteration_levy_steps = []

        for _ in range(population_size):
            parent = random.choice(population)
            candidate = cuckoo_movement(parent, instance, beta, step_scale)
            iteration_levy_steps.append(abs(levy_flight(beta)))

            comparison_index = random.randrange(population_size)
            if candidate.fitness < population[comparison_index].fitness:
                population[comparison_index] = candidate
                iteration_improvements += 1
                improvements += 1

        population.sort(key=lambda nest: nest.fitness)
        if population[0].fitness < best_nest.fitness:
            best_nest = population[0]

        number_to_abandon = int(abandonment_prob * population_size)
        for index in range(number_to_abandon):
            population[-(index + 1)] = generate_random_nest(instance)
            abandonments += 1

        population.sort(key=lambda nest: nest.fitness)
        if population[0].fitness < best_nest.fitness:
            best_nest = population[0]

        average_fitness = float(np.mean([nest.fitness for nest in population]))
        diversity = calculate_population_diversity(population)
        best_fitness_history.append(best_nest.fitness)
        avg_fitness_history.append(average_fitness)
        diversity_history.append(diversity)
        levy_step_history.append(float(np.mean(iteration_levy_steps)))

        if verbose and (iteration % 50 == 0 or iteration == max_iterations - 1):
            print(
                f'Iteration {iteration + 1:3d}: '
                f'Best-so-far = {best_nest.fitness:8.0f}, '
                f'Population average = {average_fitness:8.0f}, '
                f'Fitness diversity = {diversity:.3f}, '
                f'Accepted replacements = {iteration_improvements}'
            )

    runtime = time.time() - start_time
    convergence_iteration = int(np.argmin(best_fitness_history))
    initial_fitness = best_fitness_history[0]
    total_improvement = initial_fitness - best_nest.fitness
    improvement_percentage = (
        100.0 * total_improvement / initial_fitness
        if not math.isclose(initial_fitness, 0.0)
        else 0.0
    )

    results = {
        'best_nest': best_nest,
        'best_fitness': best_nest.fitness,
        'convergence_iteration': convergence_iteration,
        'total_improvement': total_improvement,
        'improvement_percentage': improvement_percentage,
        'runtime': runtime,
        'best_fitness_history': best_fitness_history,
        'avg_fitness_history': avg_fitness_history,
        'diversity_history': diversity_history,
        'levy_step_history': levy_step_history,
        'total_improvements': improvements,
        'total_abandonments': abandonments,
        'final_population': population,
        'parameters': {
            'population_size': population_size,
            'max_iterations': max_iterations,
            'abandonment_prob': abandonment_prob,
            'beta': beta,
            'step_scale': step_scale,
        },
    }

    if verbose:
        print('\n' + '=' * 60)
        print('CUCKOO SEARCH RESULTS')
        print('=' * 60)
        print(f'Runtime: {runtime:.3f} seconds')
        print(f'Best-so-far fitness: {best_nest.fitness:.0f}')
        print(f'Improvement from initial best: {improvement_percentage:.1f}%')
        print(f'Best-so-far first reached at iteration: {convergence_iteration}')
        print(f'Accepted replacements: {improvements}')
        print(f'Abandoned nests: {abandonments}')
        print(f'Final population fitness diversity: {diversity_history[-1]:.3f}')

    return best_nest, results


best_nest, cs_results = cuckoo_search(
    instance,
    population_size=25,
    max_iterations=500,
    abandonment_prob=0.25,
    beta=1.5,
    step_scale=0.1,
    verbose=True,
)

assert math.isclose(
    best_nest.fitness,
    calculate_nest_fitness(best_nest, instance),
    rel_tol=1e-12,
    abs_tol=1e-9,
)
assert all(
    later <= earlier
    for earlier, later in zip(
        cs_results['best_fitness_history'],
        cs_results['best_fitness_history'][1:],
    )
)
assert len(cs_results['best_fitness_history']) == 501
assert len(cs_results['avg_fitness_history']) == 501
assert len(cs_results['diversity_history']) == 501
assert len(cs_results['levy_step_history']) == 500

In [ ]:
def visualize_cuckoo_search_results(
    results: Dict, instance: CrossDockInstance
):
    """Plot convergence, fitness diversity, sampled steps, and final fitness."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle('Cuckoo Search with Lévy Flights', fontsize=16, fontweight='bold')

    iterations = range(len(results['best_fitness_history']))
    ax = axes[0, 0]
    ax.plot(iterations, results['best_fitness_history'], label='Best-so-far fitness')
    ax.plot(iterations, results['avg_fitness_history'], '--', label='Population average')
    convergence = results['convergence_iteration']
    ax.plot(
        convergence,
        results['best_fitness_history'][convergence],
        'o',
        label=f'Best first reached (iteration {convergence})',
    )
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Fitness (volume-distance units)')
    ax.set_title('Best-so-far and population-average fitness')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(iterations, results['diversity_history'], color='green')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Fitness coefficient of variation')
    ax.set_title('Population fitness diversity')
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    levy_history = results['levy_step_history']
    ax.plot(range(1, len(levy_history) + 1), levy_history, color='purple')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Mean absolute sampled Lévy step')
    ax.set_title('Sampled Lévy-step magnitudes by iteration')
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    final_fitnesses = [nest.fitness for nest in results['final_population']]
    ax.hist(final_fitnesses, bins=15, color='skyblue', edgecolor='black')
    ax.axvline(
        results['best_fitness'],
        color='red',
        label=f"Best-so-far: {results['best_fitness']:.0f}",
    )
    ax.set_xlabel('Fitness (volume-distance units)')
    ax.set_ylabel('Number of nests')
    ax.set_title('Final population fitness distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_cuckoo_search_results(cs_results, instance)
print('\nBest solution found by this run:')
print(f'Origin assignments: {best_nest.origin_assignments}')
print(f'Destination assignments: {best_nest.destination_assignments}')
print(f'Fitness: {best_nest.fitness:.0f} volume-distance units')

In [ ]:
def estimate_tail_index(data: List[float]) -> float:
    """Estimate a Pareto-style tail index with a Hill estimator."""
    positive_data = sorted((value for value in data if value > 0), reverse=True)
    if len(positive_data) < 2:
        return float('inf')

    k = max(2, len(positive_data) // 10)
    threshold = positive_data[k]
    if threshold <= 0:
        return float('inf')
    log_sum = sum(math.log(value / threshold) for value in positive_data[:k])
    if math.isclose(log_sum, 0.0):
        return float('inf')
    return float(k / log_sum)


def analyze_levy_flight_behavior(beta: float = 1.5, num_samples: int = 10000) -> Dict:
    """Summarize sampled absolute Lévy steps and their empirical tail."""
    if num_samples < 2:
        raise ValueError('At least two samples are required for this analysis.')

    levy_steps = np.array([levy_flight(beta) for _ in range(num_samples)])
    absolute_steps = np.abs(levy_steps)
    small_steps = int(np.sum(absolute_steps < 0.1))
    medium_steps = int(np.sum((absolute_steps >= 0.1) & (absolute_steps < 1.0)))
    large_steps = int(np.sum(absolute_steps >= 1.0))
    tail_index = estimate_tail_index(absolute_steps.tolist())

    print('\nLÉVY-FLIGHT STEP ANALYSIS')
    print(f'Parameter beta: {beta}')
    print(f'Sample count: {num_samples}')
    print(f'Mean absolute step: {np.mean(absolute_steps):.4f}')
    print(f'Standard deviation: {np.std(absolute_steps):.4f}')
    print(f'Median absolute step: {np.median(absolute_steps):.4f}')
    print(f'95th percentile: {np.percentile(absolute_steps, 95):.4f}')
    print(f'99th percentile: {np.percentile(absolute_steps, 99):.4f}')
    print(f'Steps below 0.1: {100 * small_steps / num_samples:.1f}%')
    print(f'Steps from 0.1 up to 1.0: {100 * medium_steps / num_samples:.1f}%')
    print(f'Steps of at least 1.0: {100 * large_steps / num_samples:.1f}%')
    print(f'Empirical Hill tail-index estimate: {tail_index:.3f}')
    print(
        'The empirical estimate describes this finite sample; it is not '
        'a proof of the distribution’s theoretical tail exponent.'
    )

    return {
        'levy_steps': levy_steps,
        'absolute_steps': absolute_steps,
        'tail_index': tail_index,
    }


levy_analysis = analyze_levy_flight_behavior(beta=1.5, num_samples=10000)
absolute_steps = levy_analysis['absolute_steps']
positive_steps = absolute_steps[absolute_steps > 0]
sorted_steps = np.sort(positive_steps)
cumulative = np.arange(1, len(sorted_steps) + 1) / len(sorted_steps)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Sampled Lévy-Flight Behavior (beta = 1.5)', fontsize=16)

ax = axes[0, 0]
ax.hist(positive_steps, bins=50, color='purple', alpha=0.7, edgecolor='black')
ax.set_xscale('log')
ax.set_xlabel('Absolute step size (log scale)')
ax.set_ylabel('Frequency')
ax.set_title('Absolute step-size distribution')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(sorted_steps, cumulative, color='blue')
ax.set_xscale('log')
ax.set_xlabel('Absolute step size (log scale)')
ax.set_ylabel('Empirical cumulative probability')
ax.set_title('Empirical cumulative distribution')
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
tail_count = max(2, len(sorted_steps) // 5)
tail_data = sorted_steps[-tail_count:]
tail_probabilities = (
    np.arange(1, tail_count + 1) / (tail_count + 1)
)
theoretical_values = (1 / (1 - tail_probabilities)) ** (1 / 1.5)
ax.scatter(theoretical_values, tail_data, alpha=0.6, label='Sample tail quantiles')
ax.plot(
    theoretical_values,
    theoretical_values,
    'b--',
    label='Reference line',
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Pareto reference quantiles (log scale)')
ax.set_ylabel('Sample tail quantiles (log scale)')
ax.set_title('Tail-quantile diagnostic')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
sample_steps = absolute_steps[:500]
ax.plot(range(1, len(sample_steps) + 1), sample_steps, color='green')
ax.axhline(
    np.mean(sample_steps),
    color='red',
    linestyle='--',
    label=f'Sample mean: {np.mean(sample_steps):.3f}',
)
ax.set_xlabel('Sample index')
ax.set_ylabel('Absolute step size')
ax.set_title('First 500 sampled step magnitudes')
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Descriptive comparison with baselines

The following experiment samples random feasible assignments and runs the notebook's Hill Climbing baseline from multiple random starting points. These are descriptive comparisons on the same instance, not optimality proofs. The Cuckoo Search value is the best result from one run, whereas the baseline summaries describe sets of sampled outcomes; therefore, a direct difference should not be interpreted as a controlled statistical test.

In [ ]:
def compare_cuckoo_search_with_baselines(instance: CrossDockInstance) -> Dict:
    """Compare one Cuckoo Search run with random and Hill Climbing outcomes."""
    number_random_solutions = 100
    random_costs = [
        generate_random_nest(instance).fitness
        for _ in range(number_random_solutions)
    ]

    def simple_hill_climbing(
        problem: CrossDockInstance, max_iterations: int = 1000
    ) -> Nest:
        current = generate_random_nest(problem)
        best = current
        for _ in range(max_iterations):
            neighbor = cuckoo_movement(
                current, problem, beta=1.5, step_scale=0.05
            )
            if neighbor.fitness < current.fitness:
                current = neighbor
                if neighbor.fitness < best.fitness:
                    best = neighbor
            else:
                break
        return best

    number_hill_climbing_trials = 10
    hill_climbing_costs = [
        simple_hill_climbing(instance).fitness
        for _ in range(number_hill_climbing_trials)
    ]

    random_stats = {
        'mean': float(np.mean(random_costs)),
        'std': float(np.std(random_costs)),
        'min': float(np.min(random_costs)),
        'max': float(np.max(random_costs)),
    }
    hill_climbing_stats = {
        'mean': float(np.mean(hill_climbing_costs)),
        'std': float(np.std(hill_climbing_costs)),
        'min': float(np.min(hill_climbing_costs)),
        'max': float(np.max(hill_climbing_costs)),
    }
    cuckoo_fitness = cs_results['best_fitness']

    def percentage_difference(reference: float, candidate: float) -> float:
        if math.isclose(reference, 0.0):
            return float('nan')
        return 100 * (reference - candidate) / reference

    print('DESCRIPTIVE BASELINE COMPARISON')
    print(f'Random assignments (n={number_random_solutions}):')
    print(
        f"  Mean: {random_stats['mean']:.0f}; "
        f"standard deviation: {random_stats['std']:.0f}; "
        f"range: {random_stats['min']:.0f}–{random_stats['max']:.0f}"
    )
    print(f'Hill Climbing (n={number_hill_climbing_trials}):')
    print(
        f"  Mean: {hill_climbing_stats['mean']:.0f}; "
        f"standard deviation: {hill_climbing_stats['std']:.0f}; "
        f"range: {hill_climbing_stats['min']:.0f}–{hill_climbing_stats['max']:.0f}"
    )
    print(f'Cuckoo Search best from one run: {cuckoo_fitness:.0f}')
    print(
        'Difference from random mean: '
        f"{percentage_difference(random_stats['mean'], cuckoo_fitness):.1f}%"
    )
    print(
        'Difference from Hill Climbing mean: '
        f"{percentage_difference(hill_climbing_stats['mean'], cuckoo_fitness):.1f}%"
    )
    print(
        'These descriptive differences do not establish statistical '
        'significance or general superiority.'
    )

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Descriptive Algorithm Comparison', fontsize=16)

    ax = axes[0, 0]
    ax.hist(random_costs, bins=20, alpha=0.5, label='Random assignments')
    ax.hist(hill_climbing_costs, bins=10, alpha=0.5, label='Hill Climbing')
    ax.axvline(cuckoo_fitness, color='red', linewidth=2, label='Cuckoo Search best')
    ax.set_xlabel('Fitness (volume-distance units)')
    ax.set_ylabel('Number of outcomes')
    ax.set_title('Outcome distributions and single-run result')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    methods = ['Random', 'Hill Climbing', 'Cuckoo Search']
    means = [random_stats['mean'], hill_climbing_stats['mean'], cuckoo_fitness]
    bests = [random_stats['min'], hill_climbing_stats['min'], cuckoo_fitness]
    deviations = [random_stats['std'], hill_climbing_stats['std'], 0.0]
    positions = np.arange(len(methods))
    width = 0.25
    ax.bar(positions - width, means, width, label='Mean or single-run best')
    ax.bar(positions, bests, width, label='Best sampled outcome')
    ax.bar(positions + width, deviations, width, label='Standard deviation')
    ax.set_xticks(positions)
    ax.set_xticklabels(methods)
    ax.set_ylabel('Fitness (volume-distance units)')
    ax.set_title('Summary measures (not equivalent estimands)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    random_mean = random_stats['mean']
    improvements = [
        0.0,
        percentage_difference(random_mean, hill_climbing_stats['mean']),
        percentage_difference(random_mean, cuckoo_fitness),
    ]
    bars = ax.bar(methods, improvements, color=['gray', 'green', 'red'])
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Difference from random mean (%)')
    ax.set_title('Descriptive difference (positive means lower fitness)')
    for bar, value in zip(bars, improvements):
        ax.annotate(
            f'{value:.1f}%',
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords='offset points',
            ha='center',
        )
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    coefficients = [
        100 * random_stats['std'] / random_stats['mean'],
        100 * hill_climbing_stats['std'] / hill_climbing_stats['mean'],
        0.0,
    ]
    ax.bar(methods, coefficients, color=['blue', 'green', 'red'])
    ax.set_ylabel('Coefficient of variation (%)')
    ax.set_title('Variation within sampled baselines')
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

    return {
        'random_stats': random_stats,
        'hill_climbing_stats': hill_climbing_stats,
        'cuckoo_fitness': cuckoo_fitness,
        'difference_from_random_mean_pct': percentage_difference(
            random_stats['mean'], cuckoo_fitness
        ),
        'difference_from_hill_climbing_mean_pct': percentage_difference(
            hill_climbing_stats['mean'], cuckoo_fitness
        ),
    }


comparison_results = compare_cuckoo_search_with_baselines(instance)

## Parameter sensitivity

Each experiment below varies one algorithm setting while holding the other listed settings fixed. Because each run is stochastic and uses one seeded random stream, differences reflect both parameter choices and random draws. They are useful for inspecting this instance, not as universal parameter rankings. The main-run settings remain unchanged.

In [ ]:
def parameter_sensitivity_analysis(instance: CrossDockInstance) -> Dict:
    """Run the specified one-factor sensitivity experiments."""
    population_sizes = [10, 15, 25, 35, 50]
    abandonment_probs = [0.1, 0.15, 0.25, 0.35, 0.5]
    beta_values = [1.0, 1.2, 1.5, 1.8, 2.0]
    step_scales = [0.05, 0.1, 0.15, 0.2, 0.3]

    population_results = []
    for population_size in population_sizes:
        _, result = cuckoo_search(
            instance,
            population_size=population_size,
            max_iterations=300,
            abandonment_prob=0.25,
            beta=1.5,
            step_scale=0.1,
            verbose=False,
        )
        population_results.append({
            'population_size': population_size,
            'best_fitness': result['best_fitness'],
            'improvement_pct': result['improvement_percentage'],
            'runtime': result['runtime'],
            'diversity_final': result['diversity_history'][-1],
        })

    abandonment_results = []
    for abandonment_prob in abandonment_probs:
        _, result = cuckoo_search(
            instance,
            population_size=25,
            max_iterations=300,
            abandonment_prob=abandonment_prob,
            beta=1.5,
            step_scale=0.1,
            verbose=False,
        )
        abandonment_results.append({
            'abandonment_prob': abandonment_prob,
            'best_fitness': result['best_fitness'],
            'improvement_pct': result['improvement_percentage'],
            'total_abandonments': result['total_abandonments'],
        })

    beta_results = []
    for beta_value in beta_values:
        _, result = cuckoo_search(
            instance,
            population_size=25,
            max_iterations=300,
            abandonment_prob=0.25,
            beta=beta_value,
            step_scale=0.1,
            verbose=False,
        )
        beta_results.append({
            'beta': beta_value,
            'best_fitness': result['best_fitness'],
            'improvement_pct': result['improvement_percentage'],
            'avg_levy_step': float(np.mean(result['levy_step_history'])),
        })

    step_results = []
    for step_scale in step_scales:
        _, result = cuckoo_search(
            instance,
            population_size=25,
            max_iterations=300,
            abandonment_prob=0.25,
            beta=1.5,
            step_scale=step_scale,
            verbose=False,
        )
        step_results.append({
            'step_scale': step_scale,
            'best_fitness': result['best_fitness'],
            'improvement_pct': result['improvement_percentage'],
            'total_improvements': result['total_improvements'],
        })

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Cuckoo Search Parameter Sensitivity', fontsize=16)

    plot_specs = [
        (axes[0, 0], population_results, 'population_size', 'Population size',
         'Population-size sensitivity', 'diversity_final', 'Final fitness diversity'),
        (axes[0, 1], abandonment_results, 'abandonment_prob', 'Abandonment probability',
         'Abandonment sensitivity', 'total_abandonments', 'Total abandoned nests'),
        (axes[1, 0], beta_results, 'beta', 'Lévy parameter beta',
         'Lévy-parameter sensitivity', 'avg_levy_step', 'Mean sampled step magnitude'),
        (axes[1, 1], step_results, 'step_scale', 'Step scale',
         'Step-scale sensitivity', 'total_improvements', 'Accepted replacements'),
    ]

    for ax, records, x_key, x_label, title, second_key, second_label in plot_specs:
        x_values = [record[x_key] for record in records]
        fitness_values = [record['best_fitness'] for record in records]
        second_values = [record[second_key] for record in records]
        ax.plot(x_values, fitness_values, 'o-', color='blue', label='Best fitness')
        ax.set_xlabel(x_label)
        ax.set_ylabel('Best fitness (volume-distance units)', color='blue')
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        twin = ax.twinx()
        twin.plot(x_values, second_values, 's--', color='darkorange', label=second_label)
        twin.set_ylabel(second_label, color='darkorange')
        lines = ax.get_lines() + twin.get_lines()
        ax.legend(lines, [line.get_label() for line in lines], loc='best')

    fig.tight_layout()
    plt.show()

    return {
        'population_sensitivity': population_results,
        'abandonment_sensitivity': abandonment_results,
        'beta_sensitivity': beta_results,
        'step_scale_sensitivity': step_results,
    }


sensitivity_results = parameter_sensitivity_analysis(instance)

## Summary and interpretation

The returned assignment is the best feasible nest recorded during the Cuckoo Search run. Its fitness is the sum of every origin-destination flow multiplied by the distance between the doors assigned to that origin and destination. A lower value indicates less modeled volume-weighted distance. The history plot records best-so-far fitness, while average fitness and fitness diversity describe the evolving population; these measures answer different questions.

A stochastic run can produce different assignments and fitness values when the random seed or settings change. The best result found is not an optimality certificate. The baseline and sensitivity plots are descriptive evidence for this instance, not proof of general superiority, statistical significance, or scalability to a particular facility size. Capacities and one-to-one assignments are checked for every evaluated nest. Additional operational requirements can be represented by extending the model and its feasibility checks, while keeping the search procedure within the same Cuckoo Search framework.

In [ ]:
def tier3_summary_and_conclusions() -> None:
    """Print a concise, data-driven summary of the current experiments."""
    parameters = cs_results['parameters']
    print('TIER 3: CUCKOO SEARCH WITH LÉVY FLIGHTS')
    print('=' * 55)
    print(
        f"Instance: {len(instance.origins)} origins, "
        f"{len(instance.destinations)} destinations, "
        f"{len(instance.inbound_doors)} inbound doors, "
        f"{len(instance.outbound_doors)} outbound doors"
    )
    print(
        f"Settings: population={parameters['population_size']}, "
        f"iterations={parameters['max_iterations']}, "
        f"abandonment probability={parameters['abandonment_prob']}, "
        f"beta={parameters['beta']}, step scale={parameters['step_scale']}"
    )
    print(f"Best fitness from this run: {cs_results['best_fitness']:.0f} volume-distance units")
    print(f"Improvement from the initial population best: {cs_results['improvement_percentage']:.1f}%")
    print(f"Best-so-far first reached at iteration: {cs_results['convergence_iteration']}")
    print(f"Runtime: {cs_results['runtime']:.3f} seconds")
    print(f"Accepted replacements: {cs_results['total_improvements']}")
    print(f"Abandoned nests: {cs_results['total_abandonments']}")
    print(f"Final population fitness diversity: {cs_results['diversity_history'][-1]:.3f}")
    print('\nBest origin assignments:')
    for origin, door in best_nest.origin_assignments.items():
        print(f'  {origin} -> {door}')
    print('Best destination assignments:')
    for destination, door in best_nest.destination_assignments.items():
        print(f'  {destination} -> {door}')
    print('\nInterpretation:')
    print(
        '  This is the best feasible assignment encountered by the '
        'configured stochastic search, not a proven global optimum.'
    )
    print(
        '  Baseline and sensitivity results are specific to this instance '
        'and these experiment settings.'
    )


tier3_summary_and_conclusions()